# Module 7: Fashion Feature Extraction
## Deep CNN Feature Maps, L2-Normalized Embeddings & Similarity Matching

This notebook demonstrates:
1. Loading the deep feature extraction engine (`FashionFeatureExtractor`).
2. Extracting dense, L2-normalized 1,280-dimensional visual embeddings.
3. Computing visual cosine similarity between matching and contrasting garments.
4. Visualizing intermediate 2D convolutional filter activation maps.

In [ ]:
import sys
from pathlib import Path

# Ensure project root in sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
import seaborn as sns

from src.feature_extractor import FashionFeatureExtractor
from src.config import CLEANED_METADATA_CSV, EMBEDDING_DIM

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 6)

### 1. Initialize Feature Extractor Engine

In [ ]:
extractor = FashionFeatureExtractor(backbone="mobilenet_v2", weights="imagenet")
print(f"Feature Extractor Backbone : {extractor.backbone_name}")
print(f"Embedding Dimensionality   : {extractor.embedding_dim}")

df = pd.read_csv(CLEANED_METADATA_CSV)
print(f"Total catalog items ready  : {len(df):,}")

### 2. Extract L2-Normalized Visual Embeddings
Visualizing the embedding vector and confirming unit norm ($\Vert v \Vert_2 = 1.0$).

In [ ]:
sample_row = df[df["canonical_category"] == "Shirt"].iloc[0]
vec = extractor.extract_features(sample_row["image_path"], l2_normalize=True)

print(f"Garment : {sample_row['productDisplayName']}")
print(f"Shape   : {vec.shape}")
print(f"L2 Norm : {np.linalg.norm(vec):.6f} (Unit hypersphere projection)")

plt.figure(figsize=(14, 3))
plt.plot(vec[:200], color="royalblue", lw=1.2)
plt.title(f"First 200 Feature Dimensions of {sample_row['productDisplayName']}", fontsize=12, fontweight="bold")
plt.xlabel("Feature Dimension Index")
plt.ylabel("Normalized Activation Value")
plt.tight_layout()
plt.show()

### 3. Visual Cosine Similarity Matrix Across Garments
Comparing two similar tops against bottomwear and footwear.

In [ ]:
sample_items = [
    df[df["canonical_category"] == "T-Shirt"].iloc[0],
    df[df["canonical_category"] == "T-Shirt"].iloc[1],
    df[df["canonical_category"] == "Jeans"].iloc[0],
    df[df["canonical_category"] == "Sneakers"].iloc[0],
]

image_paths = [row["image_path"] for row in sample_items]
labels = [f"{row['canonical_category']}\n({row['id']})" for row in sample_items]

embeddings = extractor.extract_batch(image_paths, l2_normalize=True)
sim_matrix = np.dot(embeddings, embeddings.T)

fig, axes = plt.subplots(1, 2, figsize=(15, 6), gridspec_kw={"width_ratios": [1, 1.2]})

# Display garment thumbnails
thumb_grid = np.zeros((112, 112 * len(sample_items), 3), dtype=np.uint8)
for idx, p in enumerate(image_paths):
    img = Image.open(p).resize((112, 112))
    thumb_grid[:, idx * 112 : (idx + 1) * 112] = np.array(img)
axes[0].imshow(thumb_grid)
axes[0].axis("off")
axes[0].set_title("Compared Garments", fontsize=12, fontweight="bold")

# Similarity heatmap
sns.heatmap(sim_matrix, annot=True, fmt=".3f", cmap="Blues", xticklabels=labels, yticklabels=labels, ax=axes[1], vmin=0, vmax=1)
axes[1].set_title("Pairwise Visual Cosine Similarity Matrix", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

### 4. Convolutional Feature Map Activation Visualizer
Visualizing spatial filter responses across different convolutional channels.

In [ ]:
test_path = df[df["canonical_category"] == "Dress"].iloc[0]["image_path"]
feature_maps = extractor.extract_intermediate_feature_maps(test_path)
print(f"Feature Maps Shape: {feature_maps.shape} (Height, Width, Num_Filters)")

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
axes = axes.flatten()

# Show original
orig = Image.open(test_path).resize((224, 224))
axes[0].imshow(orig)
axes[0].set_title("Original Dress Image", fontsize=10, fontweight="bold")
axes[0].axis("off")

# Show 7 filter activation channels
for idx in range(1, 8):
    f_idx = idx * 12
    if f_idx < feature_maps.shape[-1]:
        fmap_channel = feature_maps[:, :, f_idx]
        axes[idx].imshow(fmap_channel, cmap="inferno")
        axes[idx].set_title(f"Conv Filter #{f_idx}", fontsize=10, fontweight="bold")
    axes[idx].axis("off")

plt.suptitle("Intermediate Spatial Feature Activations (Contours, Silhouettes, Patterns)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()